<a href="https://colab.research.google.com/github/fidelgarz67/HumanTraffickingDashboard/blob/main/Milestone3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Modules needed
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.utils import class_weight
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.model_selection import train_test_split
import shap
import warnings
warnings.filterwarnings('ignore')
import json

In [ ]:
# Load datatset
dataset = pd.read_csv('./the_global_k_anon_dataset.csv', sep=';', low_memory=False)
dataset = dataset.drop(dataset.columns[0], axis=1)

# Filter only for US data
top_exploitation_countries = ['US', 'RU', 'PH', 'ID', 'KH']
dataset = dataset[dataset['CountryOfExploitation'].isin(top_exploitation_countries)]

# Focus Feature Selection on Sexual Exploitation
dataset = dataset[[
    # Demographic Features
    'gender', 'ageBroad', 'citizenship',
    'isSexualExploit','majorityStatus', 'majorityStatusAtExploit',

    # Means of control
    'meansOfControlSexualAbuse',
    'meansOfControlPsychologicalAbuse', 'meansOfControlThreats',
    'meansOfControlFalsePromises', 'meansOfControlWithholdsDocuments',
    'meansOfControlRestrictsMovement', 'meansOfControlTakesEarnings',
    'meansOfControlPsychoactiveSubstances',

    # Recuitment or Entry
    'RecruiterRelationship',
    'recruiterRelationIntimatePartner', 'recruiterRelationFriend',
    'recruiterRelationFamily', 'isAbduction', 'majorityEntry',

    # Contextual Features
    'isSlaveryAndPractices', 'isForcedMarriage', 'CountryOfExploitation',

    #Target
    'isSexualExploit']]

# Are there any Missing Values
dataset.isna().sum().sort_values(ascending=False)
(dataset.isna().mean() * 100).sort_values(ascending=False)

# Clean up ageBroad double hypen
age_map = {
    '0--8': '0-8',
    '9--17': '9-17',
    '18--20': '18-20',
    '21--23': '21-23',
    '24--26': '24-26',
    '27--29': '27-29',
    '30--38': '30-38',
    '39--47': '39-47',
    '48+': '48-56',
    '57+': '57+',
    'Unknown': 'Unknown'
}

dataset['ageBroad'] = dataset['ageBroad'].replace(age_map)

# Clean up missing or duplicate columns/data
binary_cols = [
    'meansOfControlSexualAbuse','meansOfControlPsychologicalAbuse',
    'meansOfControlThreats','meansOfControlFalsePromises',
    'meansOfControlWithholdsDocuments','meansOfControlRestrictsMovement',
    'meansOfControlTakesEarnings','meansOfControlPsychoactiveSubstances',
    'recruiterRelationIntimatePartner','recruiterRelationFriend',
    'recruiterRelationFamily','isAbduction','isSlaveryAndPractices',
    'isForcedMarriage']

nominal_col = ['gender','citizenship','RecruiterRelationship']

age_cols = age_encoder.get_feature_names_out(['ageBroad'])

ordinal_col = {
    'majorityStatus': ['Unknown', 'Minor', 'Adult'],
    'majorityStatusAtExploit': ['Unknown', 'Minor', 'Adult'],
    'majorityEntry': ['Unknown', 'Minor', 'Adult']}

# Drop records where we cannot identify sexual exploitation
dataset = dataset.dropna(subset=['isSexualExploit'])
dataset['isSexualExploit'] = dataset['isSexualExploit'].astype(int)

dataset[binary_cols] = dataset[binary_cols].fillna(0)

cat_cols = nominal_col + age_col + list(ordinal_col.keys()) + ['CountryOfExploitation']
dataset[cat_cols] = dataset[cat_cols].fillna('Unknown')

# Encoding for the categorical features
ordinal_encoder = OrdinalEncoder(categories=list(ordinal_col.values()))

dataset[list(ordinal_col.keys())] = ordinal_encoder.fit_transform(
    dataset[list(ordinal_col.keys())])

# Encoding for ageBoard
age_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

age_encoded = age_encoder.fit_transform(dataset[['ageBroad']])
age_cols = age_encoder.get_feature_names_out(['ageBroad'])

age_df = pd.DataFrame(age_encoded, columns=age_cols, index=dataset.index)

dataset = dataset.drop(columns=['ageBroad'])
dataset = pd.concat([dataset, age_df], axis=1)

# Encoding for the nominal/ordinal features
one_hot_col = nominal_col + ['CountryOfExploitation']
one_hot = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

one_hot_encoded = one_hot.fit_transform(dataset[one_hot_col])
ohe_feature_names = one_hot.get_feature_names_out(one_hot_col)
one_hot_encoded_df = pd.DataFrame(one_hot_encoded, columns=ohe_feature_names,
                                  index=dataset.index)

dataset = dataset.loc[:, ~dataset.columns.duplicated()]

# Define target and independent variables
X = pd.concat([dataset[binary_cols + list(ordinal_col.keys())], one_hot_encoded_df], axis=1)
y = dataset['isSexualExploit']


# Understand the structure of the dataset (Sanity Check)
print(X.shape)
print(X.dtypes)
print(X.isna().sum().sum())
print('Distribution of: ', y.value_counts(normalize=True))

# Balance the target variable
class_weights = class_weight.compute_class_weight(class_weight = 'balanced',
                                                  classes=np.unique(y), y=y)
class_weights_dict = {0: class_weights[0], 1: class_weights[1]}
print(class_weights_dict)


# # Categorical Distribution
# dataset['gender'].value_counts().plot(kind='bar')
# plt.title("Gender Distribution")
# plt.show()

# dataset['ageBroad'].value_counts().plot(kind='bar')
# plt.title("Age Broad Distribution")
# plt.show()

# dataset['citizenship'].value_counts().head(15).plot(kind='bar')
# plt.title("Citizenship Distribution (Top 15)")
# plt.show()

# dataset['isForcedLabour'].value_counts().plot(kind='bar')
# plt.title("Forced Labour Indicator Distribution")
# plt.show()

# dataset['isSexualExploit'].value_counts().plot(kind='bar')
# plt.title("Sexual Exploitation Indicator Distribution")
# plt.show()

# dataset['CountryOfExploitation'].value_counts().head(10).plot(kind='bar')
# plt.title("Top 10 Countries of Exploitation")
# plt.xticks(rotation=45)
# plt.show()

# # EDA Numerical Data
# dataset.describe()

(53915, 74)
meansOfControlSexualAbuse           float64
meansOfControlPsychologicalAbuse    float64
meansOfControlThreats               float64
meansOfControlFalsePromises         float64
meansOfControlWithholdsDocuments    float64
                                     ...   
CountryOfExploitation_ID            float64
CountryOfExploitation_KH            float64
CountryOfExploitation_PH            float64
CountryOfExploitation_RU            float64
CountryOfExploitation_US            float64
Length: 74, dtype: object
0
Distribution of:  isSexualExploit
1    0.767263
0    0.232737
Name: proportion, dtype: float64
{0: np.float64(2.1483503347146957), 1: np.float64(0.6516667875359586)}


In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Build an ANN
model = Sequential([
    Dense(128, activation='relu', input_dim=X_train.shape[1]),Dropout(0.3),
    Dense(64, activation='relu'), Dropout(0.2),
    Dense(32, activation='relu'),

    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])


In [ ]:
# Train the model
history = model.fit(X_train, y_train,
    validation_split=0.2, epochs=10, batch_size=32,
    class_weight=class_weights_dict, verbose=1)

Epoch 1/10
1079/1079 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9067 - auc: 0.9446 - loss: 0.2870 - val_accuracy: 0.9492 - val_auc: 0.9753 - val_loss: 0.1489
Epoch 2/10
1079/1079 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9434 - auc: 0.9716 - loss: 0.1941 - val_accuracy: 0.9469 - val_auc: 0.9763 - val_loss: 0.1696
Epoch 3/10
1079/1079 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9448 - auc: 0.9752 - loss: 0.1839 - val_accuracy: 0.9528 - val_auc: 0.9779 - val_loss: 0.1552
Epoch 4/10
1079/1079 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9465 - auc: 0.9756 - loss: 0.1823 - val_accuracy: 0.9482 - val_auc: 0.9774 - val_loss: 0.1780
Epoch 5/10
1079/1079 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9494 - auc: 0.9760 - loss: 0.1771 - val_accuracy: 0.9500 - val_auc: 0.9773 - val_loss: 0.1746
Epoch 6/10
1079/1079 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9473 - auc: 0.9775 - loss: 0.1738 - val_accuracy: 0.9506 - val_auc: 0.9788 - val_loss: 0.1544
Epoch 7/10
1079/1079 ━━━━━━━

In [ ]:
# Evaluate the model
loss, accuracy, auc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}, Accuracy: {accuracy:.4f}, AUC: {auc:.4f}")

y_pred_proba = model.predict(X_test)

# SHAP Explainer
X_background = X_train.sample(1000, random_state=42).values
explainer = shap.GradientExplainer(model, X_background)

X_test_array = X_test.values
shap_values = explainer.shap_values(X_test_array)
shap_values = np.squeeze(shap_values)

shap_df = pd.DataFrame(shap_values, columns=X_test.columns, index=X_test.index)
# print(shap_values.shape)
# print(shap_df.shape)


Test Loss: 0.1458, Accuracy: 0.9505, AUC: 0.9744
337/337 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [ ]:
# View top feature per Country
country = 'US'
country_mask = X_test[f'CountryOfExploitation_{country}'] == 1
shap_country = shap_df[country_mask]

# Mean absolute SHAP values per feature
top_features = shap_country.abs().mean().sort_values(ascending=False)
print(top_features.head(5))

gender_Female                       0.065078
gender_Male                         0.039516
meansOfControlTakesEarnings         0.033300
citizenship_US                      0.024707
meansOfControlPsychologicalAbuse    0.022949
dtype: float64


In [ ]:
# Create an output for the FrontEnd
def get_country_shap_summary(country, shap_df, X_test, top_n=5):
    mask = X_test[f'CountryOfExploitation_{country}'] == 1
    shap_country = shap_df[mask]

    mean_abs_shap = (
        shap_country
        .abs()
        .mean()
        .sort_values(ascending=False)
        .head(top_n)
    )

    return {
        "nation": country,
        "coordinates": country_coordinates[country],
        "top_5_features": {
            i: {
                "feature": feature,
                "importance": float(mean_abs_shap.loc[feature])
            }
            for i, feature in enumerate(mean_abs_shap.index)
        }
    }

country_coordinates = {
    'US': {'lat': 37.0902, 'lng': -95.7129},
    'RU': {'lat': 61.5240, 'lng': 105.3188},
    'PH': {'lat': 12.8797, 'lng': 121.7740},
    'ID': {'lat': -0.7893, 'lng': 113.9213},
    'KH': {'lat': 12.5657, 'lng': 104.9910}}

countries = ['US', 'RU', 'PH', 'ID', 'KH']

results = [get_country_shap_summary(country, shap_df, X_test)
    for country in countries]

# print(json.dumps(results, indent=2))
with open("country_shap_summary.json", "w") as f:
    json.dump(results, f, indent=2)